# TD6 — Exercice 2 : Variogramme 2D sur une image

**Traitement statistique, géostatistique et approche systèmes — ENSG 2A (S7)**

*Auteurs: Guillaume Caumon, Oumayma Razik*

Cet exercice étudie la variable **Values** sur une **grille image 2D (Voxet 100×100)**,
à partir du fichier `image_2D_grille.csv` (10 000 cellules). **Nature de la donnée** : `Values` est une **variable régionalisée standardisée** (moyenne ≈ 0, variance ≈ 1), fournie sur une **grille 2D complète** (10 000 cellules). Donnée pédagogique servant à étudier la structure spatiale lorsque la variable est connue partout.

## 1. Chargement des données

Lecture du fichier `image_2D_grille.csv` (grille 2D, 10 000 cellules).

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gstlearn as gl
import gstlearn.plot as gp

sys.path.append(os.path.abspath(os.path.join("..")))
import geostat_utils as gsutils

# ---- Chargement des données (Exercice 2 : grille image 2D) ----
image_2D = pd.read_csv('../data/image_2D_grille.csv')
valeurs_grille = image_2D['Values'].to_numpy()
display(image_2D)

results = gsutils.ResultsDir("TD6_outputs_vario2D")

## 2. Visualisation des données

Chaque cellule (ou pixel) correspond à une valeur de la propriété **Values**, sur .La colormap **PRGn** (Purple-Green) est utilisée. Commençons par afficher l'image.

In [ ]:
nx_map, ny_map = 100, 100
grid_image = valeurs_grille.reshape(ny_map, nx_map)
plt.figure(figsize=(7, 6))
plt.imshow(grid_image, cmap='PRGn', origin='lower',
           vmin=-1.8, vmax=1.8)
plt.colorbar(label='Values')
plt.title('Grille image 2D — Voxet 100×100', fontweight='bold')
plt.xlabel('X (cellules)')
plt.ylabel('Y (cellules)')
plt.tight_layout()
results.save_fig(f"ex2_image.png", dpi=150, bbox_inches="tight")
plt.show()

Et regardons à présent la distribution des valeurs

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(grid_image.flatten(), bins=30, color='salmon', edgecolor='black', alpha=0.7)
plt.title("Histogram of Values")
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.grid(True, alpha=0.3)
plt.show()

## 3. Création des données au format gstlearn (grille 2DMap)

In [ ]:
# ---- Création de la base gstlearn depuis la grille 2DMap ----
# ---- Paramètres du variogramme 2D ----
gl.defineDefaultSpace(gl.ESpaceType.RN, 2)
taille_pixel_x  = taille_pixel_y = 1
db = gl.DbGrid.create(
    [nx_map, ny_map],    
    [taille_pixel_x, taille_pixel_y],
    [0.0, 0.0]           
)
db.addColumns(valeurs_grille, 'Values', gl.ELoc.Z)
print(db)

## 4. Variogramme expérimental omnidirectionnel en 2D

Comme en une dimension, il convient de **choisir le nombre de pas de calcul (`nlag`), la distance unitaire (`dlag`) et la tolérance (`toldis`)** avant de pouvoir afficher le variogramme.

Attention, le calcul peut prendre du temps si vous ne choisissez pas bien les paramètres !

In [ ]:
number_of_lags =
lag_distance =
params_vario_2dmap = gl.VarioParam.createOmniDirection(
    nlag=number_of_lags,
    dlag=lag_distance,
    toldis=0.5
)
vario_2d_expe_omni = gl.Vario(params_vario_2dmap)
vario_2d_expe_omni.compute(db)
variance = vario_2d_expe_omni.getVar(0,0)

gsutils.plot_1D_variogram(vario_2d_expe_omni, title=f"Variogramme expérimental omnidirectionnel",
                          filename=f"ex1_variogramme_expérimental_omnidirectionnel_2D.png")

## 5. Ajustement automatique d'un modèle isotrope

Ajustons à présent un modèle à ce variogramme empirique

In [ ]:
# ---- Ajustement modèle Sphérique ----
model_fit_isotropic = gl.Model()
model_fit_isotropic.fit(vario_2d_expe_omni, types=[gl.ECov.NUGGET,gl.ECov.EXPONENTIAL,gl.ECov.SPHERICAL])

gsutils.plot_1D_variogram(vario_2d_expe_omni, title=f"Variogramme omnidirectionnel",
                          filename=f"ex1_variogramme_expérimental_omnidirectionnel_2D.png", model=model_fit_isotropic)
print(model_fit_isotropic)

**Que pensez-vous de l'ajustement ?**

**Le modèle de variogramme vous semble-t-il satisfaisant ?**


## 6. Carte variographique (Variogram Map)

Pour capturer les directions principales d'anisotropie, nous allons à présent calculer une carte variographique, qui donne les valeurs du variogramme dans toutes les directions et distances (discrétisées)

In [ ]:
variogram_map = gl.db_vmap(db)
fig, ax = gp.init(1, 2, figsize=(14, 7), flagEqual=True)
ax[0].raster(variogram_map, flagLegend=True, cmap="jet", vmin=0, vmax=variance * 1.5)
ax[0].decoration(title="Carte variographique")

#-----carte du nombre de paires -----
ax[1].raster(variogram_map, name="VMAP.Values.Nb", flagLegend=True, cmap="jet")
ax[1].decoration(title="Nombre de paires")
plt.tight_layout()
plt.show()

## 7. Variogramme directionnel (anisotropie)

** Dans quelles directions calculer les deux variogramme empiriques correspondant aux portées minimales et maximales ?**

Trois nouveaux paramètres sont requis pour répondre à cette question:
  - Pour l'orientation, on peut soit utiliser un vecteur `codir` [dx, dy] qui donne la direction choisie, soit utiliser un angle `angle2D` (convention mathématique: 0 suivant x (vers l'Est), 90 suivant y (vers le nord): angle2D = 90° - Azimut),
  - `tolang` est la tolérance angulaire autour de cette direction
  - `cylrad` est la tolérance maximale autorisée dans la direction orthogonale à la direction considérée


In [ ]:
#---- Variogramme directionnel en deux directions ----
number_of_lags =
lag_distance =
azimut_main =
azimut_min =
dir_main = gl.DirParam.create(
    nlag=number_of_lags,
    dlag=lag_distance,
    tolang=10,
    cylrad=3,
    angle2D = 90 - azimut_main
)
# Direction 45° — direction perpendiculaire
dir_min = gl.DirParam.create(
    nlag=number_of_lags,
    dlag=lag_distance,
    tolang=10,
    cylrad=3,
    angle2D = 90 - azimut_min
)
params_vario_dir = gl.VarioParam()
params_vario_dir.addDir(dir_main)
params_vario_dir.addDir(dir_min)

variogram_dir = gl.Vario(params_vario_dir)
variogram_dir.compute(db)

ax = gp.varmod(variogram_dir, flagLegend=True)
ax.set_ylim(0, variogram_dir.getVar() * 1.5)
plt.xlabel("Distance (m)")
plt.ylabel(r"$\gamma(h)$")
plt.title(f"Variogramme directionnel d'azimuts {azimut_main}° et {azimut_min}°")
plt.show()

### Calage de modèles de variogrammes

Au vu de l'anisotropie, on peut maintenant caler un modèle de variogramme en s'appuyant sur les deux variogrammes directionnels

In [ ]:
variogram_types = [gl.ECov.NUGGET,gl.ECov.EXPONENTIAL,gl.ECov.SPHERICAL]
variogram_model_2directions = gl.Model()
variogram_model_2directions.fit(variogram_dir, types=variogram_types)
ax = gp.varmod(variogram_dir, variogram_model_2directions, flagLegend=True)

print(variogram_model_2directions)

On peut également caler un modèle directement sur la carte variographique

In [ ]:
variogram_model_all_directions = gl.Model()
variogram_model_all_directions.fitFromVMap(variogram_map, types=variogram_types)
print(variogram_model_all_directions)

Pour valider ce modèle, nous allons à présent superposer ce modèle sur les deux variogrammes directionnels expérimentaux, et l'afficher sur la carte variographique

**Le résultat est-il compatible avec les variogrammes directionnels ?**

In [ ]:
gp.varmod(variogram_dir, variogram_model_all_directions, flagLegend=True)

variogram_model_all_directions.buildVmapOnDbGrid(variogram_map) # Creates
fig, ax = gp.init(1, 2, figsize=[8, 8], flagEqual=True)
ax[0].raster(variogram_map)
ax[0].decoration(title="Modèle de variogramme calé \n  sur la carte variographique")

variogram_model_2directions.buildVmapOnDbGrid(variogram_map) # Creates
ax[1].raster(variogram_map)
ax[1].decoration(title="Modèle de variogramme calé \n sur les deux variogrammes directionnels")

**Le résultat est-il compatible avec les variogrammes directionnels ?**


## Bonus: La nuée variographique

Pour comprendre le variogramme, nous pouvons calculer tous les écarts quadratiques entre toutes les paires de points, et afficher leur densité en fonction de la distance. C'est ce qu'on appelle la nuée variographique.
Le variogramme expérimental est une moyenne locale de ces points.

In [ ]:
# Nuée variographique
varioParamOmni_cloud = gl.VarioParam.createOmniDirection(
    nlag=number_of_lags,
    dlag=lag_distance
)
vario_cloud = gl.db_vcloud(db, varioParamOmni_cloud)
print(vario_cloud)

#
gp.plot(vario_cloud, "Cloud.Values", cmap="BrBG")
ax = plt.gca()
ax.varmod(vario_2d_expe_omni, color="black", linewidth=2)
plt.title("Nuée variographique omnidirectionnelle")
plt.xlabel("Distance (km)")
plt.ylabel(r"$\gamma(h)$")
plt.ylim(0, 4)
plt.show()